In [13]:
# Import Libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split




In [14]:
# import csv file into pd.dataframe
df = pd.read_csv("Housing.csv")

In [15]:
# Clean the housing data
clean_df = df.copy()
rows_before = len(clean_df)

# Clean whitespace & trim text columns
clean_df = clean_df.replace(r"^\s*$", pd.NA, regex=True)
text_columns = clean_df.select_dtypes(include=["str"]).columns
for column in text_columns:
    clean_df[column] = clean_df[column].str.strip()

# Parse dates & convert all numeric fields
clean_df["date"] = pd.to_datetime(clean_df["date"], errors="coerce")
numeric_columns = [
    "price", "bedrooms", "bathrooms", "sqft_living", "sqft_lot", "floors",
    "waterfront", "view", "condition", "sqft_above", "sqft_basement",
    "yr_built", "yr_renovated"
]
for column in numeric_columns:
    clean_df[column] = pd.to_numeric(clean_df[column], errors="coerce")

# Removing duplicates & invalid records
clean_df = clean_df.drop_duplicates()
clean_df = clean_df.dropna(subset=["price", "bedrooms", "bathrooms", "sqft_living", "sqft_lot"])
clean_df = clean_df[
    (clean_df["price"] > 0)
    & (clean_df["bedrooms"] > 0)
    & (clean_df["bathrooms"] > 0)
    & (clean_df["sqft_living"] > 0)
    & (clean_df["sqft_lot"] > 0)
]


print(f"Rows before cleaning: {rows_before}")
print(f"Rows after cleaning: {len(clean_df)}")
print(f"Removed rows: {rows_before - len(clean_df)}")
print(f"Missing values remaining: {clean_df.isna().sum().sum()}")
print(clean_df.describe())


Rows before cleaning: 4140
Rows after cleaning: 4089
Removed rows: 51
Missing values remaining: 0
                             date         price     bedrooms    bathrooms  \
count                        4089  4.089000e+03  4089.000000  4089.000000   
mean   2014-06-10 18:10:18.048422  5.593763e+05     3.395207     2.157679   
min           2014-05-02 00:00:00  7.800000e+03     1.000000     0.750000   
25%           2014-05-27 00:00:00  3.250000e+05     3.000000     1.750000   
50%           2014-06-12 00:00:00  4.645000e+05     3.000000     2.250000   
75%           2014-06-25 00:00:00  6.600000e+05     4.000000     2.500000   
max           2014-07-10 00:00:00  2.659000e+07     8.000000     6.750000   
std                           NaN  5.839753e+05     0.896196     0.775224   

        sqft_living      sqft_lot       floors   waterfront         view  \
count   4089.000000  4.089000e+03  4089.000000  4089.000000  4089.000000   
mean    2135.050379  1.467577e+04     1.513695     0.006

In [16]:
# Inspecting the cleaned dataset
print("Shape:", clean_df.shape)
print("\nColumn names:")
print(clean_df.columns.tolist())
print("\nData types:")
print(clean_df.dtypes)
print("\nMissing values by column:")
print(clean_df.isna().sum())


Shape: (4089, 18)

Column names:
['date', 'price', 'bedrooms', 'bathrooms', 'sqft_living', 'sqft_lot', 'floors', 'waterfront', 'view', 'condition', 'sqft_above', 'sqft_basement', 'yr_built', 'yr_renovated', 'street', 'city', 'statezip', 'country']

Data types:
date             datetime64[us]
price                   float64
bedrooms                float64
bathrooms               float64
sqft_living               int64
sqft_lot                  int64
floors                  float64
waterfront                int64
view                      int64
condition                 int64
sqft_above                int64
sqft_basement             int64
yr_built                  int64
yr_renovated              int64
street                      str
city                        str
statezip                    str
country                     str
dtype: object

Missing values by column:
date             0
price            0
bedrooms         0
bathrooms        0
sqft_living      0
sqft_lot         0
floors  

In [17]:
# preparing features and split the data
model_df = clean_df.copy()
model_df["yr_renovated"] = model_df["yr_renovated"].fillna(0)

X = model_df.drop(columns=["price", "date", "street", "country"])
y = model_df["price"]

X = pd.get_dummies(X, columns=["city", "statezip"], dtype=int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))
print("Features:", X.shape[1])

Training rows: 3271
Testing rows: 818
Features: 132


In [ ]:
# Random Forest Model
model = RandomForestRegressor(
    n_estimators=300,
    random_state=42,
    n_jobs=-1,
    min_samples_leaf=2,
)
model.fit(X_train, y_train)

predictions = model.predict(X_test)
mae = mean_absolute_error(y_test, predictions)
rmse = np.sqrt(mean_squared_error(y_test, predictions))
r2 = r2_score(y_test, predictions)

print(f"Mean absolute error: ${mae:,.0f}")
print(f"Root mean squared error: ${rmse:,.0f}")
print(f"R-squared: {r2:.3f}")

# Example: predicting the price of one held-out house.
example_features = X_test.iloc[[0]]
example_prediction = model.predict(example_features)[0]
print(f"Example predicted price: ${example_prediction:,.0f}")

Mean absolute error: $110,805
Root mean squared error: $205,959
R-squared: 0.502
Example predicted price: $539,267
